In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
path2save = "/data/users/04_share_reanalysis_results/01_aml/aml_data_scclone2dr/data_preprocessed_for_scclone2dr"
path2saveraw = "/data/users/04_share_reanalysis_results/01_aml/aml_data_scclone2dr/raw_data"

In [ ]:
df = pd.read_csv('aml_online.csv')
samplesonline = df['SampleID'].values

In [ ]:
samplesonline = np.sort(samplesonline)
samplesonline = [sample for sample in samplesonline if not(sample in [ 'DOROBOF', 'DOROFEG'])]

# Saving mapping from samples to patients

In [ ]:
filepath = "/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv"

info = pd.read_csv(filepath, sep='\t', header=0)
samples2keep = []
samples2keep_info = []
for sample in info['sampleID']:
    sample_name = sample.split('-')[0]
    if not(sample_name in samplesonline):
        print(sample, sample)
    else:
        samples2keep.append(sample_name)
        samples2keep_info.append(sample)
print(samples2keep)


print('Size info before filering', info.shape)
info = info[info['sampleID'].isin(samples2keep_info)]
print('Size info after filering', info.shape)
info = info[['sampleID', 'patient_id', 'tissue_type']]
info = info.reset_index()
info.to_csv(os.path.join(path2save, 'info_cohort.csv'), sep='\t', index=False)

assert (info['sampleID'].apply(lambda x: x.split('-')[0] in samplesonline)).values.all()

# Saving metacells

In [ ]:
cellmode = "metacells_hallmarks_phenograph"
df = pd.read_csv(f"/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{cellmode}/clone_infos.csv", index_col=0)

In [ ]:
columns2keep = ["clonelabel", "clonecategory"]
for col in df.columns:
    if col not in columns2keep:
        sample_name =  col.split("_")[1].split('-')[0]
        if sample_name in samplesonline:
            columns2keep.append(col)
print(columns2keep)
print('Size before filtering', df.shape)
df = df[columns2keep]
print("Size after filtering", df.shape)

In [ ]:
df.to_csv(os.path.join(path2save, cellmode, "clone_infos.csv"))

In [ ]:
os.makedirs(os.path.join(path2save, cellmode, "sample2data"), exist_ok=True)

In [ ]:
files = os.listdir(f"/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{cellmode}/sample2data")
import shutil


samplenamesrna2file = {file.split('-')[0]:file for file in files}
sampleskeptinRNA = []
if True:
    for sample, file in samplenamesrna2file.items():
        if sample in samplesonline:
            src = f"/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{cellmode}/sample2data/{file}"
            dst = os.path.join(path2save, cellmode, "sample2data", file)
            shutil.copyfile(src, dst)
            sampleskeptinRNA.append(sample)

In [ ]:
pathgsva = '/data/users/04_share_reanalysis_results/01_aml/04_metacells_gsva/hallmarks/'
os.makedirs(os.path.join(path2saveraw, "gsva"), exist_ok=True)
files = os.listdir(pathgsva)
samplenamesgsva2file = {file.split('-')[0]:file for file in files}
for sample, file in samplenamesgsva2file.items():
    if sample in samplesonline and (sample in sampleskeptinRNA):
        src = os.path.join(pathgsva, file)
        dst = os.path.join(path2saveraw, "gsva", file)
        shutil.copyfile(src, dst)

In [ ]:
path_MC = '/data/users/04_share_reanalysis_results/01_aml/03_metacells_atypical_removed/'
os.makedirs(os.path.join(path2saveraw, "metacells"), exist_ok=True)
files = os.listdir(path_MC)

file2samplenameMC = {file:file.split('-')[0] for file in files}
for file,sample in file2samplenameMC.items():
    if sample in samplesonline and (sample in sampleskeptinRNA):
        src = os.path.join(path_MC, file)
        dst = os.path.join(path2saveraw, "metacells", file)
        shutil.copyfile(src, dst)

# Pharmacoscopy

In [ ]:
df = pd.read_csv('/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv')

print("Size before filtering", df.shape)
df = df[df['SampleID'].isin(samplesonline)]
print("Size after filtering", df.shape)
df.to_csv(os.path.join(path2save, "pharmacoscopy.csv"), index=False)